# K-ABENA — Transformer NLP : PyTorch ↔ TensorFlow

K-ABENA pour les Transformers peut s'appliquer à **deux niveaux** :

| Niveau | Granularité | Perte filtrée | Usage |
|--------|-------------|---------------|-------|
| **Séquence** | 1 valeur par document | BCE/CE moyenne du document | Classification de texte |
| **Token** | 1 valeur par token | CE par token | Génération, LLM fine-tuning |

Ce notebook couvre le **niveau séquence** (classification de texte) — le plus commun en pratique.

**Architecture** : Embedding → Transformer Encoder → Pooling → Dense → Softmax

---

In [ ]:
import numpy as np

# Dataset simulé (embeddings pré-calculés, dim=64)
# En pratique : remplacer par BERT embeddings, fastText, etc.
np.random.seed(42)
N, SEQ_LEN, DIM = 800, 32, 64

# Simule des séquences de tokens avec embeddings
X_np = np.random.randn(N, SEQ_LEN, DIM).astype(np.float32)
# 3 classes de sentiment : négatif / neutre / positif
y_np = (np.random.rand(N) * 3).astype(int)

split = int(0.8 * N)
X_train, X_test = X_np[:split], X_np[split:]
y_train, y_test = y_np[:split], y_np[split:]

# Hyperparamètres K-ABENA
K_ABENA  = 0.40   # textes bien classifiés (perte < 0.40) = mineurs
N_ABENA  = 0.30   # conserver 30% des textes faciles
EPOCHS   = 30
D_MODEL  = 64
N_HEADS  = 4
N_LAYERS = 2
N_CLASSES = 3

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Classes: {N_CLASSES}")

---
# VERSION PYTORCH — Transformer Encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from kabena_ml.integrations.torch_utils import kabena_filter_torch

# ── Modèle Transformer PyTorch ────────────────────────────────────────────
class TransformerClassifier_PT(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, n_classes=3):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head    = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x : (batch, seq_len, d_model)
        encoded = self.encoder(x)          # (batch, seq_len, d_model)
        pooled  = encoded.mean(dim=1)      # mean pooling → (batch, d_model)
        return self.head(pooled)           # (batch, n_classes)

model_pt  = TransformerClassifier_PT(D_MODEL, N_HEADS, N_LAYERS, N_CLASSES)
opt_pt    = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
X_pt = torch.tensor(X_train)
y_pt = torch.tensor(y_train, dtype=torch.long)
history_pt = []

# ── Boucle PyTorch + K-ABENA ──────────────────────────────────────────────
for epoch in range(EPOCHS):
    model_pt.train()
    # AVANT K-ABENA (1 ligne) :
    # loss = F.cross_entropy(model_pt(X_pt), y_pt)
    #
    # APRÈS K-ABENA (+2 lignes) :
    losses = F.cross_entropy(model_pt(X_pt), y_pt, reduction='none')  # ← 'none'
    mask   = kabena_filter_torch(losses, K=K_ABENA, N=N_ABENA)        # ← +1 ligne
    L_KA   = losses[mask].mean()

    opt_pt.zero_grad(); L_KA.backward(); opt_pt.step()
    m = mask.sum().item()
    history_pt.append({"epoch": epoch, "loss": L_KA.item(),
                        "m": m, "gain_pct": round((1-m/len(y_pt))*100)})
    if epoch % 10 == 0:
        print(f"[PT] Ep {epoch:3d} | loss={L_KA.item():.4f} | m={m}/{len(y_pt)} | gain={m//1}%")

model_pt.eval()
with torch.no_grad():
    preds_pt = model_pt(torch.tensor(X_test)).argmax(1).numpy()
    acc_pt   = (preds_pt == y_test).mean()
print(f"\n[PT] Accuracy finale : {acc_pt:.4f} | Gain moyen : {np.mean([r['gain_pct'] for r in history_pt]):.1f}%")

---
# VERSION TENSORFLOW — Transformer Encoder

In [ ]:
import tensorflow as tf
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

# ── Couche Transformer personnalisée TF ───────────────────────────────────
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.att  = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model//num_heads)
        self.ffn  = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model),
        ])
        self.ln1  = tf.keras.layers.LayerNormalization()
        self.ln2  = tf.keras.layers.LayerNormalization()
        self.drop = tf.keras.layers.Dropout(rate)

    def call(self, x, training=False):
        attn_out = self.att(x, x, training=training)
        x        = self.ln1(x + self.drop(attn_out, training=training))
        ffn_out  = self.ffn(x)
        return self.ln2(x + self.drop(ffn_out, training=training))

# ── Modèle Transformer TF (même architecture que PyTorch) ─────────────────
inputs  = tf.keras.Input(shape=(SEQ_LEN, D_MODEL))
x       = TransformerBlock(D_MODEL, N_HEADS, 128)(inputs)
x       = TransformerBlock(D_MODEL, N_HEADS, 128)(x)
x       = tf.keras.layers.GlobalAveragePooling1D()(x)  # mean pooling
outputs = tf.keras.layers.Dense(N_CLASSES)(x)
model_tf = tf.keras.Model(inputs, outputs)

# ── Option A : Callback (+1 ligne dans model.fit) ─────────────────────────
model_tf.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# AVANT K-ABENA : model_tf.fit(X_train, y_train, epochs=EPOCHS)
# APRÈS K-ABENA :
model_tf.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=len(X_train), verbose=0,
    callbacks=[KabenaCallback(K=K_ABENA, N=N_ABENA, verbose=False)]  # ← +1 ligne
)
_, acc_tf = model_tf.evaluate(X_test, y_test, verbose=0)
print(f"[TF Callback]     Accuracy : {acc_tf:.4f}")

# ── Option B : GradientTape (boucle explicite, équivalent PyTorch) ─────────
inputs2  = tf.keras.Input(shape=(SEQ_LEN, D_MODEL))
x2       = TransformerBlock(D_MODEL, N_HEADS, 128)(inputs2)
x2       = TransformerBlock(D_MODEL, N_HEADS, 128)(x2)
x2       = tf.keras.layers.GlobalAveragePooling1D()(x2)
outputs2 = tf.keras.layers.Dense(N_CLASSES)(x2)
model_tf2 = tf.keras.Model(inputs2, outputs2)

cfg     = KabenaConfig(K=K_ABENA, N=N_ABENA, task='classification')
trainer = KabenaTFTrainer(model_tf2, cfg, tf.keras.optimizers.Adam(1e-3))
trainer.fit(tf.constant(X_train), tf.constant(y_train), epochs=EPOCHS, batch_size=len(X_train))

preds2  = tf.argmax(model_tf2(X_test, training=False), axis=1).numpy()
acc_tf2 = (preds2 == y_test).mean()
print(f"[TF GradientTape] Accuracy : {acc_tf2:.4f}")

## Table de correspondance Transformer — PyTorch ↔ TensorFlow

| Composant | PyTorch | TensorFlow |
|-----------|---------|------------|
| Self-attention | `nn.MultiheadAttention(d, heads, batch_first=True)` | `tf.keras.layers.MultiHeadAttention(num_heads, key_dim)` |
| Encoder layer | `nn.TransformerEncoderLayer(d, heads, dff)` | `TransformerBlock` personnalisé (voir ci-dessus) |
| Layer Norm | `nn.LayerNorm(d)` | `tf.keras.layers.LayerNormalization()` |
| Mean pooling | `x.mean(dim=1)` | `GlobalAveragePooling1D()(x)` |
| Format input | `(batch, seq, d)` avec `batch_first=True` | `(batch, seq, d)` natif |
| **K-ABENA** | `kabena_filter_torch(losses, K, N)` | `KabenaCallback(K, N)` |

## K-ABENA niveau token (pour LLM fine-tuning)

```python
# PyTorch — K-ABENA par token
tok_losses = F.cross_entropy(
    logits.view(-1, vocab_size),
    labels.view(-1),
    reduction='none'
)  # shape: (batch * seq_len,)
mask   = kabena_filter_torch(tok_losses, K=0.25)  # filtre les tokens bien prédits
L_KA   = tok_losses[mask].mean()

# TensorFlow — K-ABENA par token
tok_losses = tf.keras.losses.sparse_categorical_crossentropy(
    labels_flat, logits_flat, from_logits=True
)  # shape: (batch * seq_len,)
mask   = trainer.filter(tok_losses)  # même logique
L_KA   = tf.reduce_mean(tf.boolean_mask(tok_losses, mask))
```